# Companion Notebook 08: Grouped-Query Attention (GQA) Head Mapping

This notebook verifies the tensor operations and mapping mechanics of **Grouped-Query Attention (GQA)**, demonstrating how Key and Value heads are duplicated to align with Query heads.


## 1. Simulating GQA Head Expansion
We simulate GQA with:
- Query heads $h_Q = 8$
- Key-Value heads $h_{KV} = 2$
- Group size $G = h_Q / h_{KV} = 4$
- Head dimension $d_{\text{head}} = 4$
- Sequence length $L = 3$
- Batch size $b = 1$


In [1]:
import torch
import numpy as np

# Set options
torch.set_printoptions(precision=4, sci_mode=False)

b = 1
L = 3
h_Q = 8
h_KV = 2
d_head = 4

# Initialize random Query, Key, Value tensors
# Shape: [batch, seq_len, num_heads, head_dim]
Q = torch.randn(b, L, h_Q, d_head)
K = torch.randn(b, L, h_KV, d_head)
V = torch.randn(b, L, h_KV, d_head)

print("Original Tensor Shapes:")
print("  Query Shape:      ", list(Q.shape))
print("  Key Shape (KV):   ", list(K.shape))
print("  Value Shape (KV): ", list(V.shape))

# Group size calculation
group_size = h_Q // h_KV
print(f"\nGroup Size (G) = {h_Q} // {h_KV} = {group_size}")


Original Tensor Shapes:
  Query Shape:       [1, 3, 8, 4]
  Key Shape (KV):    [1, 3, 2, 4]
  Value Shape (KV):  [1, 3, 2, 4]

Group Size (G) = 8 // 2 = 4


In [2]:
# Step 1: Duplicate Key and Value heads to match Query heads
# We expand Key from [b, L, h_KV, d_head] to [b, L, h_Q, d_head] by repeating along the head dimension
# Use repeat_interleave to repeat each KV head G times
K_expanded = torch.repeat_interleave(K, repeats=group_size, dim=2)
V_expanded = torch.repeat_interleave(V, repeats=group_size, dim=2)

print("Expanded Tensor Shapes:")
print("  Expanded Key:  ", list(K_expanded.shape))
print("  Expanded Value:", list(V_expanded.shape))

# Verify mapping: KV head index 0 should be duplicated for Query heads 0, 1, 2, 3
print("\nVerification of mapping:")
print("Original Key Head 0:\n", K[0, 0, 0, :].numpy())
print("Expanded Key Head 0 (maps to Query 0):\n", K_expanded[0, 0, 0, :].numpy())
print("Expanded Key Head 3 (maps to Query 3):\n", K_expanded[0, 0, 3, :].numpy())
print("Are they identical?")
print(torch.allclose(K[0, 0, 0, :], K_expanded[0, 0, 3, :]))

print("\nOriginal Key Head 1:\n", K[0, 0, 1, :].numpy())
print("Expanded Key Head 4 (maps to Query 4):\n", K_expanded[0, 0, 4, :].numpy())
print("Are they identical?")
print(torch.allclose(K[0, 0, 1, :], K_expanded[0, 0, 4, :]))


Expanded Tensor Shapes:
  Expanded Key:   [1, 3, 8, 4]
  Expanded Value: [1, 3, 8, 4]

Verification of mapping:
Original Key Head 0:
 [-2.1584244   0.5113287   1.2435269  -0.27416077]
Expanded Key Head 0 (maps to Query 0):
 [-2.1584244   0.5113287   1.2435269  -0.27416077]
Expanded Key Head 3 (maps to Query 3):
 [-2.1584244   0.5113287   1.2435269  -0.27416077]
Are they identical?
True

Original Key Head 1:
 [-0.36612085  0.8446961   0.9073812   0.6436646 ]
Expanded Key Head 4 (maps to Query 4):
 [-0.36612085  0.8446961   0.9073812   0.6436646 ]
Are they identical?
True


### Explanation of Outputs (GQA Head Expansion)
- **Shapes**: The query has 8 heads, and KV has 2. Repeating each KV head by $group\_size = 4$ expands Key to 8 heads.
- **Mapping check**: Verifies that KV head index 0 matches Expanded Key head index 0, 1, 2, 3, and KV head index 1 matches Expanded Key head index 4, proving the Grouped-Query GQA mapping mathematically.


In [3]:
# Step 2: Compute GQA self-attention using expanded heads
# Reshape tensors to isolate heads: [b, num_heads, seq_len, head_dim]
Q_t = Q.transpose(1, 2)            # [1, 8, 3, 4]
K_t = K_expanded.transpose(1, 2)   # [1, 8, 3, 4]
V_t = V_expanded.transpose(1, 2)   # [1, 8, 3, 4]

# Compute attention scores per head
scores = torch.matmul(Q_t, K_t.transpose(-2, -1)) / np.sqrt(d_head) # [1, 8, 3, 3]
weights = torch.softmax(scores, dim=-1)
output = torch.matmul(weights, V_t) # [1, 8, 3, 4]

# Transpose output back: [b, seq_len, num_heads, head_dim]
output_gqa = output.transpose(1, 2)
print("\nFinal GQA Output Shape:", list(output_gqa.shape))



Final GQA Output Shape: [1, 3, 8, 4]


### Explanation of Outputs (GQA Self-Attention Calculation)
- **Output Shape**: `[1, 3, 8, 4]` represents batch size 1, sequence length 3, query heads 8, and head dimension 4. This shows that the final attention computation has successfully integrated the grouped keys and values.
